# Exploratory questions: how Madrid's bus fleet works and how it could transition
**IE Sustainability Datathon (Kearney) · Supporting notebook 03**

Works through the six questions the team agreed on the whiteboard. It uses the same setup cells and `data_raw/` files as the main notebook. Findings feed the main notebook and the report. **This notebook never writes `File_1.csv`–`File_4.csv`.**

Sections 2–5 follow the EDA steps (overview → data preparation → features → hypotheses) and produce the clean tables the questions use. Each question is tagged **[ONCE]** (Madrid only) or **[PER CITY]** (repeated for each benchmark city), as on the board.

## Scope agreed on the whiteboard

| # | Question | Scope | Sub-questions |
|---|---|---|---|
| Q1 | How does the current fleet work? | ONCE | **Buses:** km (range), type (standard / articulated), capacity, gas / electric, age · **Lines:** electric / gas, # buses, riders, km, schedule · **Depots:** grid capacity, electrical output |
| Q2 | Apply the Madrid setup to each benchmark city | PER CITY | KPI differences · time / phases (% electrified = # zero-emission lines ÷ total lines) · gap between Madrid and the city |
| Q3 | Why were certain lines electrified first? | ONCE | Features: depot, similarity of lines, distance, traffic, # buses, riders |
| Q4 | How would we transition the buses? | ONCE | **a)** grid / depots: depot information, gap to electrification per depot, route impact / coverage · **b)** which routes are replaced: phases, route impact (buses, bus metrics) |
| Q5 | What does implementing electrification mean? | ONCE | Finance / costs: bus age and retirement, gas, labour |
| Q6 | Scenario comparison | ONCE | Depot → line · area / zones · bus ages / lifetime · distance · emissions · ridership |

**Data groups:** Madrid's existing setup (km, fleet, lines, depots and grid capacity, emissions) · benchmark cities · external signals (traffic, incline) · financials (age / retirement, gas, labour).
**Constraints:** zero-emission routes · ridership.
**Out of scope** (crossed out on the board): weather, trends / patterns, disposal, backups.

## 0 · Setup
Same helpers and fixed parameters as the main notebook (`fetch`, `assume`, `check`, `P`, GTFS helpers). Keep them identical: if you change one here, change it in notebook 01 too.

In [1]:
# Environment: runs as-is in Google Colab (Python 3.10+).
!pip -q install pandas numpy pyproj shapely folium plotly kaleido requests

import os, io, json, zipfile, datetime as dt, hashlib, textwrap
import numpy as np, pandas as pd, requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

BASE = "/content/datathon" if os.path.exists("/content") else os.path.abspath("..")
RAW, DERIVED, OUT = (os.path.join(BASE, d) for d in ("data_raw", "data_derived", "outputs"))
for d in (RAW, DERIVED, OUT):
    os.makedirs(d, exist_ok=True)

RUN_DATE = dt.date.today().isoformat()
import sys, platform
print(f"Run date {RUN_DATE} | Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print("Folders:", RAW, DERIVED, OUT, sep="\n  ")


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


Run date 2026-11-13 | Python 3.14.7 | pandas 3.1.1 | numpy 2.5.4
Folders:
  /Users/ellagarcia/Datathon/madrid-bus-electrification/data_raw
  /Users/ellagarcia/Datathon/madrid-bus-electrification/data_derived
  /Users/ellagarcia/Datathon/madrid-bus-electrification/outputs


In [2]:
# ---- Data register: every downloaded or manually obtained source is logged here.
DATA_REGISTER = []

def fetch(url, filename, note="", manual=False):
    """Download a public file into data_raw/ and log source, date, size and checksum.
    manual=True: the file was obtained by hand (registration/API); it must already be in data_raw/."""
    path = os.path.join(RAW, filename)
    if not manual and not os.path.exists(path):
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    size = os.path.getsize(path) if os.path.exists(path) else None
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()[:12] if size else None
    DATA_REGISTER.append(dict(file=filename, url=url, downloaded=RUN_DATE, bytes=size,
                              sha256_12=sha, obtained="manual" if manual else "script", note=note))
    return path

def show_register():
    display(pd.DataFrame(DATA_REGISTER))

# ---- Provenance labels (brief section 4.6): V verified, D derived, P proxy, A assumption.
ASSUMPTIONS = []

def assume(variable, value, label, evidence, impact="", sensitivity=""):
    assert label in {"V", "D", "P", "A"}, "label must be V, D, P or A"
    ASSUMPTIONS.append(dict(variable=variable, value=value, label=label, evidence=evidence,
                            impact=impact, sensitivity=sensitivity))
    return value

# ---- Visible checks: every step prints CHECK ... PASS / FAIL.
CHECKS = []

def check(name, ok, detail=""):
    CHECKS.append(dict(check=name, result="PASS" if ok else "FAIL", detail=detail))
    print(f"CHECK {'PASS' if ok else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    return ok

In [3]:
# Fixed parameters from the official brief, section 4.5. Do not hard-code these numbers anywhere else.
P = dict(
    kwh_km      = {"Standard": 1.30, "Articulated": 1.85},   # battery energy incl. HVAC
    range_km    = {"Standard": 300,  "Articulated": 240},    # effective daily range (75% of tender minimum)
    cng_kg_km   = {"Standard": 0.45, "Articulated": 0.65},
    charger_kw  = 100,
    window_h    = 7,                                          # 23:00-06:00
    charge_eff  = 0.90,
    bus_price_eur = {"Standard": 615_000, "Articulated": 800_000},  # excl. VAT
    benchmark_kwh_km = 1.30,
    service_days = 365,
    # TODO(owner: emissions) confirm value, unit and boundary against the cited page before submission
    cng_kgco2_per_kg = assume("CO2 factor CNG", 2.7175, "V", "MITECO emission factors (inventory annex VII)", "Files 2, 4"),
    grid_gco2_kwh    = assume("CO2 factor electricity Spain 2025", 108.4, "V", "Red Electrica 2025 generation emissions", "Files 1, 2, 4"),
)
display(pd.DataFrame([(k, v) for k, v in P.items()], columns=["parameter", "value"]))

,parameter,value
0,kwh_km,"{'Standard': 1.3, 'Articulated': 1.85}"
1,range_km,"{'Standard': 300, 'Articulated': 240}"
2,cng_kg_km,"{'Standard': 0.45, 'Articulated': 0.65}"
3,charger_kw,100
4,window_h,7
5,charge_eff,0.9
6,bus_price_eur,"{'Standard': 615000, 'Articulated': 800000}"
7,benchmark_kwh_km,1.3
8,service_days,365
9,cng_kgco2_per_kg,2.7175


In [4]:
# ---- Generic GTFS helpers (shared logic with the benchmark notebook).
def read_gtfs(zip_path, tables=("agency","routes","trips","stop_times","calendar","calendar_dates","shapes","feed_info")):
    feed = {}
    with zipfile.ZipFile(zip_path) as z:
        names = set(z.namelist())
        for t in tables:
            if f"{t}.txt" in names:
                feed[t] = pd.read_csv(z.open(f"{t}.txt"), dtype=str, low_memory=False)
    return feed

def active_service_ids(feed, date_str):
    """Service ids running on date_str (YYYY-MM-DD), applying calendar then calendar_dates exceptions."""
    d = pd.Timestamp(date_str); ymd = d.strftime("%Y%m%d"); dow = d.day_name().lower()
    active = set()
    if "calendar" in feed:
        c = feed["calendar"]
        m = (c["start_date"] <= ymd) & (c["end_date"] >= ymd) & (c[dow] == "1")
        active |= set(c.loc[m, "service_id"])
    if "calendar_dates" in feed:
        cd = feed["calendar_dates"]; cd = cd[cd["date"] == ymd]
        active |= set(cd.loc[cd["exception_type"] == "1", "service_id"])
        active -= set(cd.loc[cd["exception_type"] == "2", "service_id"])
    return active

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

def shape_lengths_km(feed):
    s = feed["shapes"].copy()
    for c in ("shape_pt_lat","shape_pt_lon"): s[c] = s[c].astype(float)
    s["shape_pt_sequence"] = s["shape_pt_sequence"].astype(int)
    s = s.sort_values(["shape_id","shape_pt_sequence"])
    g = s.groupby("shape_id")
    s["seg"] = haversine_km(g["shape_pt_lat"].shift(), g["shape_pt_lon"].shift(), s["shape_pt_lat"], s["shape_pt_lon"])
    return s.groupby("shape_id")["seg"].sum()

def trip_km(feed, trip_ids, dist_unit_m=None):
    """Km per trip. Preferred: shape_dist_traveled (unit must be confirmed: metres or km);
    fallback: geodesic length of the trip's shape. Prints which method each trip used."""
    st = feed["stop_times"][feed["stop_times"]["trip_id"].isin(trip_ids)]
    km = pd.Series(dtype=float)
    if "shape_dist_traveled" in st and st["shape_dist_traveled"].notna().any():
        d = st.assign(x=pd.to_numeric(st["shape_dist_traveled"], errors="coerce")).groupby("trip_id")["x"].agg(lambda v: v.max() - v.min())
        assert dist_unit_m is not None, "Confirm shape_dist_traveled unit (metres or km) and pass dist_unit_m=True/False"
        km = d / 1000 if dist_unit_m else d
    missing = set(trip_ids) - set(km.dropna().index)
    if missing and "shapes" in feed:
        lens = shape_lengths_km(feed)
        t = feed["trips"].set_index("trip_id").loc[list(missing), "shape_id"]
        km = pd.concat([km.dropna(), t.map(lens)])
    print(f"trip_km: {len(km)} trips, {len(missing)} via shape geometry fallback")
    return km

## 1 · Data
Same sources as main notebook §1. `fetch()` reuses files already in `data_raw/`.

In [5]:
# Direct file URLs, resolved from each landing page via the portal's CKAN API (brief section 4.1).
EMT = "https://datos.emtmadrid.es/dataset"
SOURCES = {
    # filename: (url, note)
    "emt_gtfs.zip": (f"{EMT}/9b23259a-4491-494b-9695-36a7709b2c12/resource/3cba2058-9833-422c-a704-bf992d31d2ee/download/gtfs_emt.zip",
                     "EMT GTFS, rolling daily feed — https://datos.madrid.es/dataset/900030-0-emt-autobus-gtfs"),
    "emt_supply_2026.csv": (f"{EMT}/03c64a21-3ee8-461a-9f9a-cb4c727a41be/resource/8400b44b-ca21-45b9-b3cf-7a15b754b739/download/2026busoferta.csv",
                     "Buses in service per day, line and hour band, 2026 — https://datos.madrid.es/dataset/300596-0-emt-lineas-autobuses-tramos"),
    "emt_fleet_history.csv": (f"{EMT}/071db69e-5e59-4ceb-a6da-92d895f56b62/resource/48d59bf3-93a8-48b7-96bd-cf78cc7c902b/download/vehicles.csv",
                     "Fleet history (vehicle, dates, fuel, size) — https://datos.madrid.es/dataset/300241-0-emt-flota-autobuses"),
    "emt_calendar.csv": (f"{EMT}/405a6ed0-9464-4d04-9edb-939031717b20/resource/b8cceab8-9173-4eec-8431-21f5e9e25834/download/calendar.csv",
                     "Operating calendar 2024-2027 — https://datos.madrid.es/dataset/900022-0-calendario-emt-autobus"),
    "emt_ridership_2026.csv": (f"{EMT}/29b27217-429f-44a0-a90a-2db7f13529b6/resource/07622d4b-6c64-43ea-a398-69134d6128bc/download/2026demdialineafranjaanual.csv",
                     "Optional: daily riders per line and hour band, 2026 — https://datos.madrid.es/dataset/900028-0-emt-demanda-autobus"),
}
PATHS = {fn: fetch(url, fn, note) for fn, (url, note) in SOURCES.items()}
show_register()

,file,url,downloaded,bytes,sha256_12,obtained,note
0,emt_gtfs.zip,https://datos.emtmadrid.es/dataset/9b23259a-44...,2026-11-13,21044007,62453fb95492,script,"EMT GTFS, rolling daily feed — https://datos.m..."
1,emt_supply_2026.csv,https://datos.emtmadrid.es/dataset/03c64a21-3e...,2026-11-13,21555684,9fdcee98ef74,script,"Buses in service per day, line and hour band, ..."
2,emt_fleet_history.csv,https://datos.emtmadrid.es/dataset/071db69e-5e...,2026-11-13,1109695,ff90c42290bd,script,"Fleet history (vehicle, dates, fuel, size) — h..."
3,emt_calendar.csv,https://datos.emtmadrid.es/dataset/405a6ed0-94...,2026-11-13,56877,b967eda5e8d5,script,Operating calendar 2024-2027 — https://datos.m...
4,emt_ridership_2026.csv,https://datos.emtmadrid.es/dataset/29b27217-42...,2026-11-13,24395594,d3bffd83ce3e,script,"Optional: daily riders per line and hour band,..."


## 2 · EDA step 1: connect the data and get an overview
Read every source exactly as published (all columns as text, nothing changed yet): size, duplicates, missing cells, then a column-by-column profile. This is what we are dealing with before any cleaning.

In [6]:
# Raw read: strings only. The calendar is read with keep_default_na=False because 'NA' is a real day-type code (step 2a).
import plotly.express as px
raw = {
    "fleet":    pd.read_csv(PATHS["emt_fleet_history.csv"], dtype=str),
    "supply":   pd.read_csv(PATHS["emt_supply_2026.csv"], sep=";", dtype=str),
    "riders":   pd.read_csv(PATHS["emt_ridership_2026.csv"], sep=";", dtype=str),
    "calendar": pd.read_csv(PATHS["emt_calendar.csv"], dtype=str, keep_default_na=False),
}

def overview(df):
    return dict(rows=len(df), columns=df.shape[1], duplicate_rows=int(df.duplicated().sum()),
                missing_cells=int(df.isna().sum().sum()), memory_mb=round(df.memory_usage(deep=True).sum() / 1e6, 1))

display(pd.DataFrame({name: overview(df) for name, df in raw.items()}).T)
with zipfile.ZipFile(PATHS["emt_gtfs.zip"]) as z:
    gtfs_rows = {n[:-4]: sum(1 for _ in z.open(n)) - 1 for n in z.namelist()}
display(pd.Series(gtfs_rows, name="rows").to_frame().T)

,rows,columns,duplicate_rows,missing_cells,memory_mb
fleet,12920.0,11.0,5675.0,0.0,7.9
supply,939717.0,5.0,0.0,0.0,247.1
riders,1026041.0,4.0,0.0,0.0,220.4
calendar,1461.0,10.0,0.0,0.0,0.8


,routes,stops,trips,frequencies,calendar_dates,calendar,agency,stop_times,shapes
rows,238,4934,86635,86599,8,5,1,2226330,194888


In [7]:
def profile(df):
    return pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "unique": df.nunique(),
                         "examples": [", ".join(map(str, df[c].dropna().unique()[:6])) for c in df.columns]})

for name, df in raw.items():
    print(f"\n=== {name}: first rows and column profile")
    display(df.head(3)); display(profile(df))


=== fleet: first rows and column profile


,vehicleId,dateIni,dateEnd,trade,model,version,typeCapacity,fuelType,accesibility,capacity,colour
0,0051,2023-04-11 00:00:00,9999-12-31 00:00:00,IRIZAR,IETRAM,0,01,EL,04,80,B
1,0052,2023-03-25 00:00:00,9999-12-31 00:00:00,IRIZAR,IETRAM,0,01,EL,04,80,B
2,0053,2023-05-05 00:00:00,9999-12-31 00:00:00,IRIZAR,IETRAM,0,01,EL,04,80,B


,dtype,missing,unique,examples
vehicleId,str,0,5351,"0051, 0052, 0053, 0054, 0055, 0056"
dateIni,str,0,1423,"2023-04-11 00:00:00, 2023-03-25 00:00:00, 2023..."
dateEnd,str,0,925,"9999-12-31 00:00:00, 2004-07-16 00:00:00, 2017..."
trade,str,0,24,"IRIZAR , IRIZAR, PEGASO , IVECO , MA..."
model,str,0,68,"IETRAM , IETRAM, 6424 , I2E , IE..."
version,str,0,12,"0, 7, 1, 2, 3, 4"
typeCapacity,str,0,6,"01 , 02 , 05 , 03 , 06 , 04"
fuelType,str,0,6,"EL , GB , GN , ET , DE , H"
accesibility,str,0,5,"04 , 01 , 03 , 4 , 02"
capacity,str,0,35,"80, 51, 77, 87, 120, 118"



=== supply: first rows and column profile


,Linea,FechaSer,TipoDiaMo,CodFranja,NumCCReal
0,001,2026-01-01,FE,7,2
1,001,2026-01-01,FE,8,3
2,001,2026-01-01,FE,9,3


,dtype,missing,unique,examples
Linea,str,0,246,"001, 002, 003, 004, 005, 006"
FechaSer,str,0,243,"2026-01-01, 2026-01-02, 2026-01-03, 2026-01-04..."
TipoDiaMo,str,0,3,"FE, LA, SA"
CodFranja,str,0,32,"7, 8, 9, 10, 11, 12"
NumCCReal,str,0,30,"2, 3, 4, 8, 10, 9"


... (2 outputs left out)

,dtype,missing,unique,examples
Fecha,str,0,273,"2026-01-01, 2026-01-02, 2026-01-03, 2026-01-04..."
Linea,str,0,254,"001, 002, 003, 004, 005, 006"
Franja,str,0,32,"H07, H08, H09, H10, H11, H12"
TotalViajeros,str,0,3243,"17, 21, 26, 76, 95, 131"



=== calendar: first rows and column profile


,Fecha,TipoDiaEs,TemporadaTG,DiaSemana,Semana,Mes,Trimestre,Clima,TemperaturaMax,TemperaturaMin
0,2027-12-31,NA,IV,V,53,12,4,BBB,0.0,0.0
1,2027-12-30,NA,IV,J,53,12,4,BBB,0.0,0.0
2,2027-12-29,NA,IV,X,53,12,4,BBB,0.0,0.0


,dtype,missing,unique,examples
Fecha,str,0,1461,"2027-12-31, 2027-12-30, 2027-12-29, 2027-12-28..."
TipoDiaEs,str,0,8,"NA, LA, FE, SA, SS, HU"
TemporadaTG,str,0,4,"IV, VN, IM, BJ"
DiaSemana,str,0,7,"V, J, X, M, L, D"
Semana,str,0,53,"53, 52, 51, 50, 49, 48"
Mes,str,0,12,"12, 11, 10, 9, 8, 7"
Trimestre,str,0,4,"4, 3, 2, 1"
Clima,str,0,5,"BBB, , Pr , SPR, LLU"
TemperaturaMax,str,0,292,"0.0, 1.2, 25.9, 27.8, 28.0, 27.7"
TemperaturaMin,str,0,245,"0.0, 0.1, 14.8, 13.4, 15.1, 13.7"


## 3 · EDA step 2: data preparation
What step 1 showed, and what we do about it:

| Issue | Where | Action |
|---|---|---|
| Text padded with spaces (`'EL '`, `'IRIZAR    '`) | fleet | strip all text |
| 44 % of rows are exact duplicates; the rest still has several versions of the same vehicle (same dates, different capacity, colour or accessibility) | fleet | drop exact duplicates; keep one record per vehicle |
| `'NA'` is a day-type code (Christmas period), not a missing value | calendar | read with `keep_default_na=False` and label it |
| `'?'` colour, blank weather, `0.0` temperatures | fleet, calendar | columns dropped (not used; weather is out of scope) |
| Hour band is `'H07'` / `'M01'` (after midnight, night lines) in ridership, `7` / `25` in supply | riders | convert to the supply scale: `H`*nn* → *nn*, `M`*nn* → 24 + *nn* |
| Spanish column names and codes | all | rename to English `snake_case`, add readable labels |
| Lines with riders but no supply records | riders vs supply | keep, but flag: they cannot get a peak-bus count |

### 3a · Reading issues

In [8]:
na_days = sorted(raw["calendar"].loc[raw["calendar"].TipoDiaEs == "NA", "Fecha"])
print(f"Calendar: {len(na_days)} days coded 'NA'; a default read turns them into missing values. Dates: {', '.join(na_days[:9])} ...")
padded = {c: int((raw["fleet"][c] != raw["fleet"][c].str.strip()).sum()) for c in raw["fleet"].columns}
print("Fleet values padded with spaces, per column:", {k: v for k, v in padded.items() if v})

Calendar: 62 days coded 'NA'; a default read turns them into missing values. Dates: 2024-01-01, 2024-01-02, 2024-01-03, 2024-01-04, 2024-01-05, 2024-01-06, 2024-01-07, 2024-12-23, 2024-12-24 ...
Fleet values padded with spaces, per column: {'trade': 12701, 'model': 10315, 'typeCapacity': 12920, 'fuelType': 12920, 'accesibility': 12920}


### 3b · Duplicate rows and records

In [9]:
fleet = raw["fleet"].apply(lambda c: c.str.strip())
n0 = len(fleet); fleet = fleet.drop_duplicates(); n1 = len(fleet)
print(f"Fleet: {n0 - n1} exact duplicate rows removed ({(n0 - n1) / n0:.0%}); {n1} records left")

key = ["vehicleId", "dateIni"]
conflicts = pd.Series({c: int(fleet.groupby(key)[c].nunique().gt(1).sum()) for c in fleet.columns if c not in key},
                      name="vehicle + start-date groups with more than one value")
display(conflicts.to_frame())

fleet["_cap"] = pd.to_numeric(fleet.capacity, errors="coerce")
fleet = fleet.sort_values("_cap", ascending=False).drop_duplicates(key).drop(columns="_cap")
assume("Fleet near-duplicate rule", "one record per vehicle, highest capacity kept", "A",
       "Near-duplicates differ mainly in capacity and colour; the few that differ in fuel are not in service today")

check("Fleet: one record per vehicle", fleet.vehicleId.is_unique, f"{len(fleet)} records, {fleet.vehicleId.nunique()} vehicles")
for name, k in {"supply": ["Linea", "FechaSer", "CodFranja"], "riders": ["Fecha", "Linea", "Franja"], "calendar": ["Fecha"]}.items():
    check(f"{name}: no duplicate keys", not raw[name].duplicated(k).any())

Fleet: 5675 exact duplicate rows removed (44%); 7245 records left


,vehicle + start-date groups with more than one value
dateEnd,0
trade,0
model,10
version,14
typeCapacity,6
fuelType,10
accesibility,130
capacity,1429
colour,361


CHECK PASS | Fleet: one record per vehicle | 5351 records, 5351 vehicles
CHECK PASS | supply: no duplicate keys


CHECK PASS | riders: no duplicate keys
CHECK PASS | calendar: no duplicate keys


### 3c · Missing values and placeholders
Counts blanks, `NaN` and `'?'` per column. Zero-rider rows are real observations (very early or late hours) and are kept.

In [10]:
def blanks(df):
    s = df.apply(lambda c: (c.isna() | c.str.strip().isin(["", "?"])).sum())
    return s[s > 0].to_dict() or "none"

for name, df in raw.items():
    print(f"{name}: {blanks(df)}")
zero_rows = int((pd.to_numeric(raw["riders"].TotalViajeros) == 0).sum())
print(f"riders: {zero_rows} line-hour rows with 0 riders (kept)")

fleet: {'colour': 1}


supply: none


riders: none
calendar: {'Clima': 571}
riders: 148 line-hour rows with 0 riders (kept)


### 3d · Rename, recode, set types and drop unused columns
Code labels that are not in a published data dictionary are logged as proxies (**P**).

In [11]:
FUEL = {"GN": "CNG", "EL": "Battery-electric", "H": "Hydrogen fuel cell", "DE": "Diesel", "ET": "Ethanol", "GB": "Unconfirmed (GB)"}
DAY = {"LA": "Working day", "SA": "Saturday", "FE": "Sunday / holiday", "NA": "Christmas period",
       "SS": "Holy Week", "HU": "Strike", "EV": "Event", "PU": "Bridge day"}
assume("Fuel code labels", FUEL, "P", "EL, GN and H match EMT's published fleet mix; DE, ET and GB inferred, no data dictionary")
assume("Zero-emission fuel codes", ("EL", "H"), "A", "Battery-electric and hydrogen only; CNG and biomethane never count (README rule)")
assume("Day-type labels", DAY, "P", "LA, SA, FE documented in the supply README; the others inferred from their dates")

fleet = (fleet.rename(columns={"vehicleId": "vehicle_id", "dateIni": "start_date", "dateEnd": "end_date", "trade": "make",
                               "typeCapacity": "size_code", "fuelType": "fuel_code"})
              .drop(columns=["version", "accesibility", "colour"]))
fleet["start_date"] = pd.to_datetime(fleet.start_date.str[:10])
fleet["end_date"] = pd.to_datetime(fleet.end_date.str[:10].replace("9999-12-31", "2262-04-11"))   # open-ended = still in service
fleet["capacity"] = pd.to_numeric(fleet.capacity).astype("Int64")
fleet["fuel"] = fleet.fuel_code.map(FUEL)
fleet["zero_emission"] = fleet.fuel_code.isin(["EL", "H"])

cal = (raw["calendar"].rename(columns={"Fecha": "date", "TipoDiaEs": "day_code", "TemporadaTG": "season_code",
                                       "DiaSemana": "weekday_code", "Semana": "week", "Mes": "month", "Trimestre": "quarter"})
          .drop(columns=["Clima", "TemperaturaMax", "TemperaturaMin"]))     # weather: out of scope (whiteboard)
cal["date"] = pd.to_datetime(cal.date)
cal["day_type"] = cal.day_code.map(DAY)

supply = raw["supply"].rename(columns={"Linea": "line_id", "FechaSer": "date", "TipoDiaMo": "day_code",
                                       "CodFranja": "hour_band", "NumCCReal": "buses_in_service"})
supply = supply.astype({"hour_band": int, "buses_in_service": int})
supply["date"] = pd.to_datetime(supply.date)

riders = raw["riders"].rename(columns={"Fecha": "date", "Linea": "line_id", "Franja": "hour_band", "TotalViajeros": "riders"})
band = riders.hour_band.str[1:].astype(int)                                      # 'H07' -> 7; 'M01' = 01:00 after midnight -> 25
riders["hour_band"] = np.where(riders.hour_band.str.startswith("M"), band + 24, band)
riders["riders"] = riders.riders.astype(int)
riders["date"] = pd.to_datetime(riders.date)
riders = riders.merge(cal[["date", "day_code"]], on="date", how="left")        # day type from EMT's calendar

agree = supply[["date", "day_code"]].drop_duplicates().merge(cal[["date", "day_code"]], on="date", suffixes=("_supply", "_calendar"))
print("Supply vs calendar day codes (rows = supply, columns = calendar):")
display(pd.crosstab(agree.day_code_supply, agree.day_code_calendar))
only_riders = sorted(set(riders.line_id) - set(supply.line_id))
print(f"Lines with riders but no supply records (kept, flagged): {only_riders}")

check("Every fleet fuel code has a label", fleet.fuel.notna().all())
check("Every calendar day has a label", cal.day_type.notna().all())
check("Every ridership date is in the calendar", riders.day_code.notna().all())
check("Hour bands share one scale (0-31)", riders.hour_band.between(0, 31).all() and supply.hour_band.between(0, 31).all())

Supply vs calendar day codes (rows = supply, columns = calendar):


day_code_calendar,EV,FE,LA,NA,SA,SS
day_code_supply,,,,,,
FE,0,37,0,3,0,3
LA,1,0,161,2,0,3
SA,0,0,0,1,31,1


Lines with riders but no supply records (kept, flagged): ['741', '809', '841', '886', '887', '888', '889', '890']
CHECK PASS | Every fleet fuel code has a label
CHECK PASS | Every calendar day has a label
CHECK PASS | Every ridership date is in the calendar
CHECK PASS | Hour bands share one scale (0-31)


np.True_

### 3e · Plot the data: distributions and outliers
Daily values per line (peak buses, riders) by day type, and capacity by fuel. Outliers are counted with the 1.5 × IQR rule and **kept**: they are real high-demand lines, not errors.

In [12]:
line_day = (supply.groupby(["line_id", "date", "day_code"]).buses_in_service.max().rename("peak_buses").reset_index()
            .merge(riders.groupby(["line_id", "date"]).riders.sum().reset_index(), on=["line_id", "date"], how="left"))

px.box(line_day, x="day_code", y="peak_buses", title="Peak buses per line and day, by day type").show()
px.box(line_day, x="day_code", y="riders", title="Riders per line and day, by day type").show()
px.histogram(fleet.drop_duplicates("vehicle_id", keep="last"), x="capacity", color="fuel", barmode="overlay",
             title="Capacity per vehicle (latest record), by fuel").show()

def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75]); return s[(s < q1 - 1.5 * (q3 - q1)) | (s > q3 + 1.5 * (q3 - q1))]
wd = line_day[line_day.day_code == "LA"]
for col in ("peak_buses", "riders"):
    o = iqr_outliers(wd[col].dropna())
    print(f"Working days, {col}: {len(o)} line-days above/below 1.5 x IQR; lines: {sorted(wd.loc[o.index, 'line_id'].unique())[:12]}")

Working days, peak_buses: 725 line-days above/below 1.5 x IQR; lines: ['019', '021', '027', '028', '034', '049', '068', '069', '133']
Working days, riders: 1285 line-days above/below 1.5 x IQR; lines: ['021', '027', '028', '031', '034', '035', '038', '049', '068', '069', '070', '138']


### 3f · Clean tables
The tables every later section uses. Saved to `data_derived/` (git-ignored) so other notebooks can read them without repeating step 2.

In [13]:
CLEAN = dict(fleet=fleet, calendar=cal, supply=supply, riders=riders)
for name, df in CLEAN.items():
    df.to_csv(os.path.join(DERIVED, f"clean_{name}.csv.gz"), index=False)
display(pd.DataFrame({name: overview(df) for name, df in CLEAN.items()}).T)
for name, df in CLEAN.items():
    print(name, list(df.columns))

,rows,columns,duplicate_rows,missing_cells,memory_mb
fleet,5351.0,10.0,0.0,0.0,1.9
calendar,1461.0,8.0,0.0,0.0,0.5
supply,939717.0,5.0,0.0,0.0,119.3
riders,1026041.0,5.0,0.0,0.0,130.3


fleet ['vehicle_id', 'start_date', 'end_date', 'make', 'model', 'size_code', 'fuel_code', 'capacity', 'fuel', 'zero_emission']
calendar ['date', 'day_code', 'season_code', 'weekday_code', 'week', 'month', 'quarter', 'day_type']
supply ['line_id', 'date', 'day_code', 'hour_band', 'buses_in_service']
riders ['date', 'line_id', 'hour_band', 'riders', 'day_code']


## 4 · EDA step 3: features
One row per line on working days. Built from supply and ridership only; the features that need other sources are listed below the heatmap.

| Feature | Meaning |
|---|---|
| `peak_buses` | max buses in service in any hour band (average over working days) |
| `bus_hours` | sum of buses in service over all hour bands: a proxy for vehicle-hours |
| `span_hours` | number of hour bands with service |
| `peak_to_average` | `peak_buses` ÷ average buses per active hour: how peaky the line is |
| `riders_per_day` | daily riders on working days |
| `riders_per_bus_hour` | productivity: riders ÷ `bus_hours` |
| `weekend_riders_ratio` | average Saturday / Sunday riders ÷ working-day riders |

In [14]:
s = supply[supply.day_code == "LA"]
per_day = s.groupby(["line_id", "date"]).agg(peak_buses=("buses_in_service", "max"), bus_hours=("buses_in_service", "sum"),
                                             span_hours=("hour_band", "nunique"))
features = per_day.groupby("line_id").mean()
rd = riders.groupby(["line_id", "date", "day_code"]).riders.sum().reset_index()
features["riders_per_day"] = rd[rd.day_code == "LA"].groupby("line_id").riders.mean()
features["weekend_riders_ratio"] = rd[rd.day_code.isin(["SA", "FE"])].groupby("line_id").riders.mean() / features.riders_per_day
features["riders_per_bus_hour"] = features.riders_per_day / features.bus_hours
features["peak_to_average"] = features.peak_buses / (features.bus_hours / features.span_hours)
features = features.round(2)

display(features.describe().round(1))
features.to_csv(os.path.join(DERIVED, "line_features.csv"))
check("Line features: every supply line has riders", features.riders_per_day.notna().mean() > 0.95,
      f"{features.riders_per_day.notna().sum()} of {len(features)} lines")

,peak_buses,bus_hours,span_hours,riders_per_day,weekend_riders_ratio,riders_per_bus_hour,peak_to_average
count,240.0,240.0,240.0,240.0,226.0,240.0,240.0
mean,8.1,112.7,17.0,6794.0,0.7,52.7,1.3
std,4.7,72.2,4.5,5862.5,0.4,22.6,0.2
min,1.0,4.0,2.7,47.0,0.0,3.4,1.0
25%,4.3,54.4,17.0,2219.1,0.4,37.8,1.2
50%,7.6,106.7,19.0,5948.9,0.5,52.6,1.2
75%,10.9,152.3,20.0,9469.0,0.7,66.9,1.4
max,25.7,359.9,21.2,32876.0,1.6,144.0,1.7


CHECK PASS | Line features: every supply line has riders | 240 of 240 lines


np.True_

In [15]:
corr = features.corr()
px.imshow(corr.round(2), text_auto=True, zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
          title="Correlation between line features (working days)").show()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print("Highly correlated pairs (|r| > 0.8), keep one of each for modelling:")
display(pairs[pairs.abs() > 0.8].round(2).to_frame("r"))

Highly correlated pairs (|r| > 0.8), keep one of each for modelling:


r
peak_buses bus_hours             0.93
           riders_per_day        0.86
bus_hours  riders_per_day        0.92
span_hours weekend_riders_ratio -0.84

**Features still to add** (other sources): daily km and average trip distance (GTFS, main notebook §4) · zero-emission status per line (evidence list: 45 electric + line 145 hydrogen) · depot (main notebook §5) · traffic and incline (external, Q3).

## 5 · EDA step 4: hypotheses
What we expect the data to show, and where each is tested.

| # | Hypothesis | Question | Data | How we test it |
|---|---|---|---|---|
| H1 | Lines electrified first are short, high-ridership lines served by depots that were converted early | Q3 | line features + ZE label + depot | compare feature distributions, electrified vs not |
| H2 | At the 2021–25 purchase pace, Madrid reaches 100 % zero-emission only in the late 2040s | Timeline, Q6 | fleet | pace calculation below |
| H3 | For 2030, depot grid connections bind before bus purchases do | Q4a | depot table, File 3 | required vs available MW per depot |
| H4 | Peaky lines (high `peak_to_average`) need more buses per rider, so they add the most charging demand per depot | Q4b, Q1 | line features, GTFS km | rank lines by buses and energy per rider |
| H5 | A large part of the CNG fleet reaches end of life by 2035, so retirements can carry most of the transition | Q5 | fleet age profile | retirements per year vs buses needed per year |
| H6 | Madrid's gap to peer cities comes from pace, not network size | Q2 | benchmark (notebook 02) | compare ZE share and years-to-target per city |

In [16]:
# Helper for the questions below: vehicles in service on a date, built on the clean fleet table.
def active_fleet(date):
    d = pd.Timestamp(date)
    a = (fleet[(fleet.start_date <= d) & (fleet.end_date > d)]
         .sort_values("start_date").drop_duplicates("vehicle_id", keep="last"))
    first = fleet.groupby("vehicle_id").start_date.min().rename("first_in_service")
    a = a.join(first, on="vehicle_id")
    a["age_years"] = (d - a.first_in_service).dt.days / 365.25
    return a

print(f"Clean fleet: {len(fleet)} records, {fleet.vehicle_id.nunique()} vehicles, last entry {fleet.start_date.max().date()}")

Clean fleet: 5351 records, 5351 vehicles, last entry 2025-06-05


## 6 · Timeline anchors (whiteboard)
| Anchor | Year | Meaning |
|---|---|---|
| Must do | computed below | Year the fleet is fully zero-emission at the **2021–25 purchase pace** (unadjusted S0) |
| Mid | 2035 | End of the File 4 horizon |
| Ideal | 2030 | Brief scenario S1: 100 % zero-emission by 31 Dec 2030 |
| Wildcard | open | Candidate: Madrid's official 2033 target (City of Madrid, Aug 2023) |

The pace here is **unadjusted** and the fleet history's last entry is mid-2025, so main notebook §3 reconciles the 31 Dec 2025 baseline against EMT's accounts; use its figure in the report.

In [17]:
snap = {d: active_fleet(d) for d in ("2020-12-31", "2025-12-31")}
pace_tbl = pd.DataFrame({d: dict(fleet_total=len(a), zero_emission=int(a.zero_emission.sum()), not_zero_emission=int((~a.zero_emission).sum()))
                         for d, a in snap.items()}).T
display(pace_tbl)

ze_per_year = assume("Zero-emission buses added per year, 2021-25 (unadjusted)",
                     (pace_tbl.loc["2025-12-31", "zero_emission"] - pace_tbl.loc["2020-12-31", "zero_emission"]) / 5,
                     "D", "Fleet history, active vehicles on 31 Dec 2020 and 31 Dec 2025", "Timeline", "Main notebook §3 reconciliation")
remaining = pace_tbl.loc["2025-12-31", "not_zero_emission"]
must_do_year = 2025 + int(np.ceil(remaining / ze_per_year))

TIMELINE = pd.DataFrame([
    ("Ideal", assume("Ideal year (S1)", 2030, "V", "Brief scenario S1")),
    ("Wildcard", assume("Wildcard year", 2033, "V", "City of Madrid zero-emission bus target, Aug 2023")),
    ("Mid", assume("Mid year", 2035, "V", "Brief File 4 horizon")),
    ("Must do", assume("Must-do year (S0 pace)", must_do_year, "D", f"{remaining} non-zero-emission buses / {ze_per_year:.0f} per year")),
], columns=["anchor", "year"])
display(TIMELINE)

check("Fleet split adds up", (pace_tbl.zero_emission + pace_tbl.not_zero_emission == pace_tbl.fleet_total).all())
check("Zero-emission fleet grew 2020-25", ze_per_year > 0, f"{ze_per_year:.1f} buses/year")

,fleet_total,zero_emission,not_zero_emission
2020-12-31,2066,80,1986
2025-12-31,2210,462,1748


,anchor,year
0,Ideal,2030
1,Wildcard,2033
2,Mid,2035
3,Must do,2048


CHECK PASS | Fleet split adds up
CHECK PASS | Zero-emission fleet grew 2020-25 | 76.4 buses/year


np.True_

## Q1 · How does the current fleet work? [ONCE]
### 1a · Buses
Fleet in service on 31 Dec 2025 by fuel and size code, with age. `typeCapacity` codes are not decoded yet (01 is by far the most common, likely standard 12 m): **TODO(owner: baseline)** confirm the code list and map to Standard / Articulated.

In [18]:
fleet25 = snap["2025-12-31"]
display(pd.crosstab(fleet25.fuel, fleet25.size_code, margins=True, margins_name="total"))
display(fleet25.groupby("fuel").agg(buses=("vehicle_id", "count"), median_capacity=("capacity", "median"),
                                    mean_age=("age_years", "mean"), oldest=("age_years", "max")).round(1))

size_code,01,02,03,04,06,total
fuel,,,,,,
Battery-electric,408,0,20,0,24,452
CNG,1650,85,0,13,0,1748
Hydrogen fuel cell,10,0,0,0,0,10
total,2068,85,20,13,24,2210


,buses,median_capacity,mean_age,oldest
fuel,,,,
Battery-electric,452,74.0,2.9,7.9
CNG,1748,78.0,6.6,15.1
Hydrogen fuel cell,10,74.0,0.9,0.9


In [19]:
px.histogram(fleet25, x="age_years", color="fuel", nbins=20, barmode="stack",
             title="Fleet in service on 31 Dec 2025: age by fuel", labels={"age_years": "age (years)"}).show()

### 1b · Lines
Top lines by working-day riders, from the feature table (section 4). **Still TODO:** electric / gas status per line (no bus-to-line link in the open data; evidence says 45 electric lines + line 145 hydrogen) and km / schedule from GTFS, which needs `frequencies.txt` expanded (main notebook §4).

In [20]:
display(features.sort_values("riders_per_day", ascending=False)
        [["peak_buses", "riders_per_day", "riders_per_bus_hour", "span_hours", "peak_to_average"]].head(15))
print(f"{len(features)} lines with supply records; {len(only_riders)} more have riders only: {only_riders}")

,peak_buses,riders_per_day,riders_per_bus_hour,span_hours,peak_to_average
line_id,,,,,
034,21.30,32876.04,100.03,20.00,1.30
881,13.36,28468.28,130.44,21.07,1.29
070,18.68,26815.64,104.24,20.00,1.45
027,21.61,25655.62,92.16,19.99,1.55
069,20.57,24421.44,78.85,20.00,1.33
068,21.19,23997.24,74.06,19.01,1.24
883,19.15,22271.10,61.88,21.20,1.13
028,18.84,21082.64,82.21,19.99,1.47
021,17.95,19667.44,73.98,19.99,1.35


240 lines with supply records; 8 more have riders only: ['741', '809', '841', '886', '887', '888', '889', '890']


### 1c · Depots
Grid capacity and electrical output per depot. Only Sanchinarro is evidenced (8 MW main + 5 MW backup); the other four need a documented proxy. Shared with main notebook §5–§6.

In [21]:
# TODO(owner: depots) depot table (name, coordinates, connection MW, evidence); reuse main notebook §5-§6 once written

## Q2 · Apply the Madrid setup to each benchmark city [PER CITY]
For Helsinki, Berlin, Paris and Brussels: KPI differences, time / phases, and the gap to Madrid. Needs notebook 02's `benchmark_gtfs_intermediate.csv` plus cited fleet and target figures.

**KPI definition from the board:** % electrified = # zero-emission lines ÷ total # lines (alongside the fleet share used in File 1).

In [22]:
# TODO(owner: benchmark) per-city KPI table: n_lines, ze_lines, pct_lines_electrified, ze_share_pct, target year, years to target at current pace; gap vs Madrid

## Q3 · Why were certain lines electrified first? [ONCE]
Compare already-electrified lines with the rest on: depot, similarity of lines, distance, traffic, # buses, riders. External signals: traffic, incline.

In [23]:
# TODO(owner: Q3) label the zero-emission lines (evidence list), join features from Q1 (lines_tbl), GTFS distance and depot; compare distributions

In [24]:
# TODO(owner: Q3) external signals: traffic (e.g. Madrid traffic intensity open data) and incline (elevation along shapes); document sources with fetch()

## Q4 · How would we transition the buses? [ONCE]
**a) Grid electrification / depots:** depot information, gap to electrification per depot, route impact / coverage.
**b) Which routes are replaced:** phases, route impact (buses, bus metrics).

In [25]:
# TODO(owner: Q4a) per depot: required MW (main notebook §6) vs connection MW -> gap; lines and buses covered

In [26]:
# TODO(owner: Q4b) phases of lines by year for each timeline anchor; buses and km converted per phase

## Q5 · What does implementing electrification mean? [ONCE]
Finance / costs: bus purchases (`P['bus_price_eur']`), age and retirement of the gas fleet, gas cost, labour.

In [27]:
# TODO(owner: Q5) retirements by year from the gas fleet's age profile (Q1); capex per timeline anchor; gas and labour cost sources

## Q6 · Scenario comparison [ONCE]
Compare the timeline anchors (2030 ideal · 2033 wildcard · 2035 mid · must-do year) on: depot → line, area / zones, bus ages / lifetime, distance, emissions, ridership.

In [28]:
# TODO(owner: Q6) one row per anchor: buses bought per year, depots needing upgrades, zones covered, avg retirement age, km and riders on zero-emission lines, CO2 avoided

## Visualisation
Board list: scenario planning (2030 / 2035 / wildcard), map and routes with lines, bus scheduling (peak / off-peak, days of the week, times), future state over time (costs, emissions, line transition). The bus-scheduling view is below; the rest are TODO.

In [29]:
by_hour = (supply.groupby(["day_code", "date", "hour_band"]).buses_in_service.sum()
           .groupby(["day_code", "hour_band"]).mean().rename("buses_in_service").reset_index())
by_hour["day_type"] = by_hour.day_code.map(DAY)
px.line(by_hour, x="hour_band", y="buses_in_service", color="day_type", markers=True,
        title="Average buses in service by hour, Jan-Aug 2026", labels={"hour_band": "hour band (24+ = after midnight)"}).show()
peak_hour = by_hour.loc[by_hour[by_hour.day_code == "LA"].buses_in_service.idxmax()]
print(f"Working-day peak: {peak_hour.buses_in_service:.0f} buses in hour band {peak_hour.hour_band}")

Working-day peak: 1692 buses in hour band 9


In [30]:
# TODO(owner: visualisation) scenario planning chart, route map coloured by status, future-state costs / emissions / line transition

## Checks, assumptions and sources

In [31]:
display(pd.DataFrame(CHECKS)); display(pd.DataFrame(ASSUMPTIONS)); show_register()

,check,result,detail
0,Fleet: one record per vehicle,PASS,"5351 records, 5351 vehicles"
1,supply: no duplicate keys,PASS,
2,riders: no duplicate keys,PASS,
3,calendar: no duplicate keys,PASS,
4,Every fleet fuel code has a label,PASS,
5,Every calendar day has a label,PASS,
6,Every ridership date is in the calendar,PASS,
7,Hour bands share one scale (0-31),PASS,
8,Line features: every supply line has riders,PASS,240 of 240 lines
9,Fleet split adds up,PASS,


,variable,value,label,evidence,impact,sensitivity
0,CO2 factor CNG,2.7175,V,MITECO emission factors (inventory annex VII),"Files 2, 4",
1,CO2 factor electricity Spain 2025,108.4,V,Red Electrica 2025 generation emissions,"Files 1, 2, 4",
2,Fleet near-duplicate rule,"one record per vehicle, highest capacity kept",A,Near-duplicates differ mainly in capacity and ...,,
3,Fuel code labels,"{'GN': 'CNG', 'EL': 'Battery-electric', 'H': '...",P,"EL, GN and H match EMT's published fleet mix; ...",,
4,Zero-emission fuel codes,"(EL, H)",A,Battery-electric and hydrogen only; CNG and bi...,,
5,Day-type labels,"{'LA': 'Working day', 'SA': 'Saturday', 'FE': ...",P,"LA, SA, FE documented in the supply README; th...",,
6,"Zero-emission buses added per year, 2021-25 (u...",76.4,D,"Fleet history, active vehicles on 31 Dec 2020 ...",Timeline,Main notebook §3 reconciliation
7,Ideal year (S1),2030,V,Brief scenario S1,,
8,Wildcard year,2033,V,"City of Madrid zero-emission bus target, Aug 2023",,
9,Mid year,2035,V,Brief File 4 horizon,,


,file,url,downloaded,bytes,sha256_12,obtained,note
0,emt_gtfs.zip,https://datos.emtmadrid.es/dataset/9b23259a-44...,2026-11-13,21044007,62453fb95492,script,"EMT GTFS, rolling daily feed — https://datos.m..."
1,emt_supply_2026.csv,https://datos.emtmadrid.es/dataset/03c64a21-3e...,2026-11-13,21555684,9fdcee98ef74,script,"Buses in service per day, line and hour band, ..."
2,emt_fleet_history.csv,https://datos.emtmadrid.es/dataset/071db69e-5e...,2026-11-13,1109695,ff90c42290bd,script,"Fleet history (vehicle, dates, fuel, size) — h..."
3,emt_calendar.csv,https://datos.emtmadrid.es/dataset/405a6ed0-94...,2026-11-13,56877,b967eda5e8d5,script,Operating calendar 2024-2027 — https://datos.m...
4,emt_ridership_2026.csv,https://datos.emtmadrid.es/dataset/29b27217-42...,2026-11-13,24395594,d3bffd83ce3e,script,"Optional: daily riders per line and hour band,..."
